# ============================================================
Day 01 — Setup, Variables, Data Types, f-strings, os.environ
============================================================
Module 01: Python + Async + FastAPI for LLM Engineering
Vidya Sankalp | Applied GenAI Engineering

WHO THIS IS FOR:
  Someone who has never written a single line of Python before.
  By the end of this file you will understand the exact same
  building blocks used in every real LLM application.

WHAT YOU WILL LEARN TODAY:
  1. How to set up a Python project (venv + pip)
  2. Variables — the four data types you use most
  3. Type hints — why IDEs and Pydantic need them
  4. f-strings — how to build prompts with runtime values
  5. os.environ + dotenv — how to load API keys safely

RUN THIS FILE:
  python modules/day01_setup_variables.py


## SECTION 0 — WHAT IS A COMMENT?


A LINE COMMENT starts with #
Python ignores everything after the # on that line.
Use them to explain WHY a line of code exists.


In [ ]:

# This is a single-line comment. Python ignores this.



A TRIPLE-QUOTED STRING like this block is used as a
documentation note (also called a docstring).
You will see these at the top of every file, class, and function.
They explain what the code does in plain English.
Python also ignores these at runtime (they are just strings).


## SECTION 1 — PROJECT SETUP (do this once before writing code)


STEP 1: Create a virtual environment
A virtual environment (venv) is an isolated Python installation.
Every project gets its own venv so package versions don't clash.

Run these commands in your terminal (NOT in Python):

  python -m venv venv                # create the venv
  source venv/bin/activate           # Mac / Linux
  venv\\Scripts\\activate              # Windows

You will see (venv) appear at the start of your terminal prompt.
That means you are inside the virtual environment.

STEP 2: Install packages
  pip install -r requirements.txt

STEP 3: Run a file
  python modules/day01_setup_variables.py


In [ ]:

# Check which Python you are running (run this after activate)
import sys
import os

print("=" * 60)
print("Python version :", sys.version)
print("Python location:", sys.executable)

# sys.prefix vs sys.base_prefix tells you if venv is active
# They are equal when NOT in a venv, different when inside one
in_venv = sys.prefix != sys.base_prefix
print("Inside venv    :", in_venv)

if not in_venv:
    print()
    print("WARNING: You are not inside a virtual environment!")
    print("  Run:  python -m venv venv")
    print("  Then: source venv/bin/activate    (Mac/Linux)")
    print("  Or:   venv\\Scripts\\activate        (Windows)")




## SECTION 2 — VARIABLES AND DATA TYPES


WHAT IS A VARIABLE?
A variable is a named container that stores a value.

  variable_name = value

Python figures out the data type automatically.
You do NOT need to declare the type first (unlike Java or C).

FOUR TYPES YOU USE EVERY DAY IN LLM ENGINEERING:
  int   — whole numbers       (token counts, retry limits, IDs)
  float — decimal numbers     (temperature, price, similarity score)
  str   — text                (prompts, model names, API keys)
  bool  — True or False       (feature flags, validation results)


In [ ]:

print()
print("─" * 60)
print("SECTION 2 — Variables and Data Types")
print("─" * 60)

# ── int (integer) ─────────────────────────────────────────────


int = whole number, no decimal point.
In LLM work: token counts, customer IDs, retry limits, port numbers.


In [ ]:

max_tokens   = 1024   # how many tokens the LLM is allowed to return
max_retries  = 3      # how many times to retry a failed API call
app_port     = 8000   # FastAPI will run on this port number
customer_id  = 1001   # customer ID from our e-commerce database

print("int examples:")
print("  max_tokens :", max_tokens)
print("  max_retries:", max_retries)
print("  customer_id:", customer_id)
print("  type(max_tokens) →", type(max_tokens))


# ── float (floating point number) ─────────────────────────────


float = number with a decimal point.
In LLM work: temperature, top_p, similarity scores, prices.

temperature controls how "creative" the LLM is:
  0.0 = fully deterministic (same input → always same output)
  1.0 = very creative / random
  0.2 = slightly creative (good for customer support, structured output)


In [ ]:

temperature          = 0.2     # LLM creativity setting
top_p                = 0.9     # nucleus sampling (leave at 0.9 for most tasks)
similarity_threshold = 0.85    # minimum cosine similarity for RAG retrieval
product_price        = 205.21  # price of Classic Monitor from products.csv

print()
print("float examples:")
print("  temperature         :", temperature)
print("  similarity_threshold:", similarity_threshold)
print("  product_price       :", product_price)
print("  type(temperature) →", type(temperature))


# ── str (string) ──────────────────────────────────────────────


str = text, always wrapped in quotes (single or double — both work).
In LLM work: prompts, model names, API endpoints, customer names.

Use triple quotes for multi-line strings:
  short_string = "one line"
  long_string  = \"\"\"
  this spans
  multiple lines
  \"\"\"


In [ ]:

model_name    = "gpt-4o"                          # which LLM to call
api_base_url  = "https://api.openai.com/v1"       # API endpoint
customer_name = "Danielle Johnson"                 # from customers.csv
order_status  = "In Transit"                       # from orders.csv
order_id      = "ORD-3042"                         # order identifier
product_name  = "Classic Monitor"                  # from products.csv

print()
print("str examples:")
print("  model_name   :", model_name)
print("  customer_name:", customer_name)
print("  order_status :", order_status)
print("  type(model_name) →", type(model_name))


# ── bool (boolean) ────────────────────────────────────────────


bool = True or False — ALWAYS capitalised in Python.
In LLM work: feature flags, streaming mode, validation toggles.


In [ ]:

stream_response = True    # send tokens to the user as they arrive
validate_output = True    # run Pydantic validation on the LLM response
is_verified     = False   # whether this review is a verified purchase

print()
print("bool examples:")
print("  stream_response:", stream_response)
print("  validate_output:", validate_output)
print("  is_verified    :", is_verified)
print("  type(stream_response) →", type(stream_response))


# ── None ──────────────────────────────────────────────────────


None = the absence of a value (like null in other languages).
Used when a variable has no value yet.

You will see None often in:
  - Optional function arguments (customer_id=None)
  - Dict values that are missing
  - Unset configuration values


In [ ]:

pending_response = None   # LLM response not yet received
session_id       = None   # session not yet created

print()
print("None example:")
print("  pending_response:", pending_response)
print("  type(None) →", type(None))




## SECTION 3 — TYPE HINTS


TYPE HINTS — telling Python (and your IDE) what type a variable holds.

Syntax:
  variable_name: type = value

Examples:
  max_tokens   : int   = 1024
  temperature  : float = 0.2
  model_name   : str   = "gpt-4o"
  stream       : bool  = True

IMPORTANT: Python does NOT enforce type hints at runtime.
  max_tokens: int = "one thousand"  ← Python allows this! No error.

But:
  - Your IDE (VS Code, PyCharm) will show a warning (red underline)
  - Pydantic DOES enforce them at runtime (Day 08)
  - Type hints make code much easier to read and maintain

In production LLM code, type hints are on every variable and function.
You will see them throughout this entire codebase.


In [ ]:

# Same variables from above, now with type hints added
max_tokens   : int   = 1024
temperature  : float = 0.2
model_name   : str   = "gpt-4o"
stream       : bool  = True

# Optional[str] means the value can be a string OR None
# We need to import Optional from the typing module
from typing import Optional

api_key      : Optional[str] = None   # not set yet — will load from .env

print()
print("─" * 60)
print("SECTION 3 — Type Hints")
print("─" * 60)
print("  max_tokens (int)  :", max_tokens)
print("  temperature (float):", temperature)
print("  model_name (str)  :", model_name)
print("  api_key (Optional):", api_key)




## SECTION 4 — f-STRINGS (the primary tool for building prompts)


WHAT IS AN f-STRING?
A way to embed Python variables directly inside a string.

Syntax: put the letter f before the opening quote,
        then use {variable_name} anywhere inside the string.

Without f-string:
  name = "Danielle"
  message = "Hello " + name + ", your order is ready."

With f-string (cleaner):
  message = f"Hello {name}, your order is ready."

WHY THIS IS THE MOST IMPORTANT THING IN DAY 01:
  Every LLM prompt is a template with variables injected at runtime.
  customer_name, order_id, order_status all come from the database.
  f-strings are how you inject them into the prompt text.


In [ ]:

print()
print("─" * 60)
print("SECTION 4 — f-strings")
print("─" * 60)

# ── Basic f-string ────────────────────────────────────────────

greeting = f"Hello {customer_name}, how can I help you today?"
print("Basic f-string:")
print(" ", greeting)


# ── f-string with number formatting ──────────────────────────


Inside {} you can add format specifiers after a colon:
  {value:,}    → add comma separator:   1024 → 1,024
  {value:.2f}  → 2 decimal places:      205.21 → $205.21
  {value:.1f}  → 1 decimal place:       0.2 → 0.2
  {value:10s}  → pad string to 10 chars (useful for tables)


In [ ]:

budget_note    = f"Budget: ${product_price:,.2f}"
tokens_note    = f"Token limit: {max_tokens:,} tokens"
temp_note      = f"Temperature: {temperature:.1f}"

print()
print("f-string number formatting:")
print(" ", budget_note)
print(" ", tokens_note)
print(" ", temp_note)


# ── Multi-line f-string (THE prompt pattern) ──────────────────


For long prompts, use triple-quote f-strings.
The \\ at the end of the opening line prevents a leading blank line.

This is how every real LLM prompt is built:
  - Fixed template text
  - {} placeholders filled with database values at runtime


In [ ]:

# These values come from the database (customers.csv, orders.csv)
# In production: they are loaded from Postgres (Day 08)
customer_name  = "Danielle Johnson"
order_id       = "ORD-3042"
order_status   = "In Transit"
product_name   = "Classic Monitor"
product_price  = 205.21

user_message = f"""\
Customer name  : {customer_name}
Order ID       : {order_id}
Order status   : {order_status}
Product ordered: {product_name} (${product_price:.2f})

The customer is asking: "Where is my order and when will it arrive?"
Please answer based only on the order status shown above.
Respond in 2 sentences maximum.


print()
print("Complete LLM user message (built with f-string):")
print("-" * 40)
print(user_message)

# ── System prompt as a multi-line string ─────────────────────


In [ ]:
The system prompt is the fixed instruction to the LLM.
It does NOT change per request — only the user message changes.

We will load this from a file in Day 04.
For now, we define it as a plain multi-line string.


system_prompt = """\
## Role
You are a customer support assistant for ShopSmart e-commerce.

## Task
Answer the customer's question using ONLY the information in the context.
You MUST NOT make up order details, prices, or dates.
You MUST keep your response under 3 sentences.

## Output Format
Plain English. Always mention the order ID in your response.


In [ ]:

print("System prompt:")
print("-" * 40)
print(system_prompt)


# ── Model configuration dict ──────────────────────────────────


A dict (dictionary) stores key-value pairs.
We will cover dicts fully in Day 03.
For now: notice that model config in LLM apps is always a dict.


In [ ]:

model_config = {
    "model"      : model_name,    # str
    "max_tokens" : max_tokens,    # int
    "temperature": temperature,   # float
    "stream"     : stream,        # bool
}

print("Model config dict:")
print(" ", model_config)




## SECTION 5 — os.environ AND python-dotenv


WHAT IS AN ENVIRONMENT VARIABLE?
A key=value pair stored OUTSIDE your code, in the shell environment.

WHY NOT HARDCODE API KEYS IN THE CODE?
  api_key = "sk-abc123xyz..."   ← NEVER DO THIS

Reasons:
  1. When you run: git add . && git commit && git push
     → your key is now public on GitHub forever
  2. Different team members have different keys
  3. Dev vs production use different keys

HOW TO DO IT CORRECTLY:
  1. Create a file called .env (note the dot at the start)
  2. Add your keys:
       OPENAI_API_KEY=sk-abc123xyz...
       DB_PASSWORD=supersecret
  3. Add .env to .gitignore (never commit it!)
  4. Load it with python-dotenv: load_dotenv()
  5. Read it with: os.environ.get("OPENAI_API_KEY")

The .env.example file in this project shows the format.
Copy it to .env and fill in your real values.


In [ ]:

from dotenv import load_dotenv   # pip install python-dotenv

# load_dotenv() reads .env file and puts values into os.environ
# MUST be called before any os.environ.get() call
# Silent if .env does not exist (no error)
load_dotenv()

print()
print("─" * 60)
print("SECTION 5 — os.environ + python-dotenv")
print("─" * 60)

# os.environ.get("KEY") → returns the value or None if not set
# os.environ.get("KEY", "default") → returns "default" if not set
# os.environ["KEY"] → raises KeyError if not set (less safe)

openai_api_key = os.environ.get("OPENAI_API_KEY")   # None if not in .env
db_host        = os.environ.get("DB_HOST", "localhost")  # default = "localhost"
db_port        = os.environ.get("DB_PORT", "5432")       # default = "5432"

print()
print("Loaded from .env (or defaults):")

if openai_api_key:
    # NEVER print the full key — show only first 4 and last 4 characters
    masked = openai_api_key[:4] + "..." + openai_api_key[-4:]
    print(f"  OPENAI_API_KEY : {masked}   ← loaded from .env")
else:
    print("  OPENAI_API_KEY : not set   ← copy .env.example to .env")

print(f"  DB_HOST        : {db_host}")
print(f"  DB_PORT        : {db_port}")


# ── Why .get() and not os.environ["KEY"] ─────────────────────


SAFE WAY:
  key = os.environ.get("OPENAI_API_KEY")
  → returns None if the variable is not set
  → your code keeps running (you handle None yourself)

UNSAFE WAY:
  key = os.environ["OPENAI_API_KEY"]
  → raises KeyError if the variable is not set
  → your entire program crashes with a confusing error message

In production code, always use .get() with a clear error message:


In [ ]:

missing_key = os.environ.get("OPENAI_API_KEY")
if missing_key is None:
    print()
    print("  How to set up your API key:")
    print("    1. cp .env.example .env")
    print("    2. Open .env in your editor")
    print("    3. Replace 'your_openai_key_here' with your real key")
    print("    4. Run this file again")




In [ ]:
# SUMMARY — what to remember from Day 01


KEY TAKEAWAYS:

1. Variables store values: name = "Danielle", price = 205.21

2. Four types: int, float, str, bool
   (+ None for "no value yet")

3. Type hints describe what a variable should hold:
   temperature: float = 0.2
   Python doesn't enforce them, but Pydantic (Day 08) does.

4. f-strings inject variables into strings:
   f"Hello {customer_name}, order {order_id} is {order_status}."

5. NEVER hardcode API keys. Use .env + os.environ.get().

6. The if __name__ == "__main__": block runs only when you execute
   this file directly. It does NOT run when you import this file.
   This is the standard Python pattern — every module file uses it.

NEXT: Day 02 — Conditions and Loops
  (how to route queries to different agents, retry failed API calls)


In [ ]:

print()
print("=" * 60)
print("Day 01 complete. Run: python modules/day02_conditions_loops.py")
print("=" * 60)
